# 02B. Composite Signal Factory

Build a small fixed-weight composite signal library from already-generated Phase 2 candidate signals. This notebook creates composite alpha candidates only; it does not score, WFV-test, stress test, freeze, portfolio construct, or use ML.

## 1. Purpose and scope

Use existing `candidate_signals_current` components and orient them with `signal_best_horizon_current` directions before building transparent fixed-weight composite candidates.

## 2. Imports and config

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.composite_quality import build_composite_quality_report
from src.composite_signals import build_composite_library, load_component_signal_panels
from src.composite_storage import COMPOSITE_TABLES, save_composite_outputs
from src.db import get_db_path, load_table
from src.run_config import make_run_id, make_run_timestamp

COMPOSITE_VERSION = 'phase2_composite_v1'
sqlite_db_path = get_db_path()

print(f'SQLite database: {sqlite_db_path}')
print(f'Composite version: {COMPOSITE_VERSION}')

SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
Composite version: phase2_composite_v1


## 3. Create run_id / timestamp

In [2]:
run_id = make_run_id(prefix='phase2_nb02b_composite')
run_timestamp = make_run_timestamp()

print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')

run_id: phase2_nb02b_composite_20260507_225304
run_timestamp: 2026-05-07 22:53:04


## 4. Load component signals and directions

In [3]:
signal_directions = load_table('signal_best_horizon_current', db_path=sqlite_db_path)

display(signal_directions[['signal_name', 'signal_direction', 'best_horizon', 'signal_strength']].head(20))

,signal_name,signal_direction,best_horizon,signal_strength
0,low_vol_strength,NEGATIVE_EDGE_REVERSE_SIGNAL,20,STRONG
1,volatility_20,POSITIVE_EDGE,20,STRONG
2,downside_vol_20,NEGATIVE_EDGE_REVERSE_SIGNAL,20,STRONG
3,volatility_60,POSITIVE_EDGE,20,STRONG
4,downside_vol_60,NEGATIVE_EDGE_REVERSE_SIGNAL,20,STRONG
5,amihud_illiq_20,POSITIVE_EDGE,20,MODERATE
6,range_expansion_failure_5,POSITIVE_EDGE,20,MODERATE
7,residual_return_vs_universe_20,NEGATIVE_EDGE_REVERSE_SIGNAL,20,MODERATE
8,ma_slope_20,NEGATIVE_EDGE_REVERSE_SIGNAL,20,MODERATE
9,reversal_20d,POSITIVE_EDGE,20,WEAK


## 5. Define composite config

In [4]:
composite_config = {
    'stress_reversal_blend_20': {
        'volatility_20': 0.60,
        'mean_reversion_20': 0.40,
    },
    'stress_pullback_blend_20': {
        'volatility_20': 0.60,
        'distance_from_ma_20': 0.40,
    },
    'vol_reversal_short_blend': {
        'volatility_20': 0.50,
        'reversal_5d': 0.25,
        'reversal_10d': 0.25,
    },
    'defensive_reversal_blend': {
        'low_vol_strength': 0.50,
        'mean_reversion_20': 0.30,
        'distance_from_ma_20': 0.20,
    },
    'defensive_momentum_reversal_blend': {
        'low_vol_strength': 0.50,
        'momentum_20': 0.50,
    },
    'balanced_stress_alpha': {
        'volatility_20': 0.40,
        'low_vol_strength': 0.30,
        'mean_reversion_20': 0.30,
    },
    'long_horizon_stress_blend': {
        'volatility_60': 0.60,
        'momentum_60': 0.40,
    },
    'short_horizon_reversal_blend': {
        'reversal_5d': 0.40,
        'reversal_10d': 0.40,
        'distance_from_ma_20': 0.20,
    },
}

component_names = sorted({component for weights in composite_config.values() for component in weights})
component_direction_map = signal_directions.set_index('signal_name')['signal_direction'].to_dict()

component_config_summary = pd.DataFrame(
    [
        {
            'composite_name': composite_name,
            'component_signal': component_signal,
            'weight': weight,
            'signal_direction': component_direction_map.get(component_signal),
        }
        for composite_name, weights in composite_config.items()
        for component_signal, weight in weights.items()
    ]
)

display(component_config_summary)

,composite_name,component_signal,weight,signal_direction
0,stress_reversal_blend_20,volatility_20,0.60,POSITIVE_EDGE
1,stress_reversal_blend_20,mean_reversion_20,0.40,POSITIVE_EDGE
2,stress_pullback_blend_20,volatility_20,0.60,POSITIVE_EDGE
3,stress_pullback_blend_20,distance_from_ma_20,0.40,POSITIVE_EDGE
4,vol_reversal_short_blend,volatility_20,0.50,POSITIVE_EDGE
5,vol_reversal_short_blend,reversal_5d,0.25,POSITIVE_EDGE
6,vol_reversal_short_blend,reversal_10d,0.25,POSITIVE_EDGE
7,defensive_reversal_blend,low_vol_strength,0.50,NEGATIVE_EDGE_REVERSE_SIGNAL
8,defensive_reversal_blend,mean_reversion_20,0.30,POSITIVE_EDGE
9,defensive_reversal_blend,distance_from_ma_20,0.20,POSITIVE_EDGE


## 6. Build composite signal library

In [5]:
component_signal_panels = load_component_signal_panels(component_names)

composite_signals, composite_metadata = build_composite_library(
    component_signal_panels=component_signal_panels,
    component_direction_map=component_direction_map,
    composite_config=composite_config,
    run_id=run_id,
    composite_version=COMPOSITE_VERSION,
)

composite_shapes = pd.Series({name: panel.shape for name, panel in composite_signals.items()}, name='shape')
display(composite_shapes)
display(composite_metadata)

stress_reversal_blend_20             (2098, 478)
stress_pullback_blend_20             (2098, 478)
vol_reversal_short_blend             (2098, 478)
defensive_reversal_blend             (2098, 478)
defensive_momentum_reversal_blend    (2098, 478)
balanced_stress_alpha                (2098, 478)
long_horizon_stress_blend            (2098, 478)
short_horizon_reversal_blend         (2098, 478)
Name: shape, dtype: object

,composite_name,n_components,component_signals,component_weights,component_directions,normalization,formula_type,run_id,composite_version,created_timestamp
0,stress_reversal_blend_20,2,"volatility_20,mean_reversion_20","volatility_20=0.6,mean_reversion_20=0.4","volatility_20=POSITIVE_EDGE,mean_reversion_20=...",component_rank_pct_centered_by_date,fixed_weight_average_of_direction_adjusted_com...,phase2_nb02b_composite_20260507_225304,phase2_composite_v1,2026-05-08 00:27:04
1,stress_pullback_blend_20,2,"volatility_20,distance_from_ma_20","volatility_20=0.6,distance_from_ma_20=0.4","volatility_20=POSITIVE_EDGE,distance_from_ma_2...",component_rank_pct_centered_by_date,fixed_weight_average_of_direction_adjusted_com...,phase2_nb02b_composite_20260507_225304,phase2_composite_v1,2026-05-08 00:27:04
2,vol_reversal_short_blend,3,"volatility_20,reversal_5d,reversal_10d","volatility_20=0.5,reversal_5d=0.25,reversal_10...","volatility_20=POSITIVE_EDGE,reversal_5d=POSITI...",component_rank_pct_centered_by_date,fixed_weight_average_of_direction_adjusted_com...,phase2_nb02b_composite_20260507_225304,phase2_composite_v1,2026-05-08 00:27:04
3,defensive_reversal_blend,3,"low_vol_strength,mean_reversion_20,distance_fr...","low_vol_strength=0.5,mean_reversion_20=0.3,dis...","low_vol_strength=NEGATIVE_EDGE_REVERSE_SIGNAL,...",component_rank_pct_centered_by_date,fixed_weight_average_of_direction_adjusted_com...,phase2_nb02b_composite_20260507_225304,phase2_composite_v1,2026-05-08 00:27:04
4,defensive_momentum_reversal_blend,2,"low_vol_strength,momentum_20","low_vol_strength=0.5,momentum_20=0.5","low_vol_strength=NEGATIVE_EDGE_REVERSE_SIGNAL,...",component_rank_pct_centered_by_date,fixed_weight_average_of_direction_adjusted_com...,phase2_nb02b_composite_20260507_225304,phase2_composite_v1,2026-05-08 00:27:04
5,balanced_stress_alpha,3,"volatility_20,low_vol_strength,mean_reversion_20","volatility_20=0.4,low_vol_strength=0.3,mean_re...","volatility_20=POSITIVE_EDGE,low_vol_strength=N...",component_rank_pct_centered_by_date,fixed_weight_average_of_direction_adjusted_com...,phase2_nb02b_composite_20260507_225304,phase2_composite_v1,2026-05-08 00:27:04
6,long_horizon_stress_blend,2,"volatility_60,momentum_60","volatility_60=0.6,momentum_60=0.4","volatility_60=POSITIVE_EDGE,momentum_60=NEGATI...",component_rank_pct_centered_by_date,fixed_weight_average_of_direction_adjusted_com...,phase2_nb02b_composite_20260507_225304,phase2_composite_v1,2026-05-08 00:27:04
7,short_horizon_reversal_blend,3,"reversal_5d,reversal_10d,distance_from_ma_20","reversal_5d=0.4,reversal_10d=0.4,distance_from...","reversal_5d=POSITIVE_EDGE,reversal_10d=POSITIV...",component_rank_pct_centered_by_date,fixed_weight_average_of_direction_adjusted_com...,phase2_nb02b_composite_20260507_225304,phase2_composite_v1,2026-05-08 00:27:04


## 7. Build composite quality report

In [6]:
composite_quality = build_composite_quality_report(composite_signals, composite_metadata)
quality_status_counts = composite_quality['status'].value_counts()

display(composite_quality.sort_values(['status', 'finite_pct', 'composite_name']))
display(quality_status_counts.rename('composite_count'))

,composite_name,n_components,n_dates,n_tickers,finite_pct,missing_pct,first_valid_date,last_valid_date,status
6,long_horizon_stress_blend,2,2098,478,0.875975,0.124025,2018-04-20,2026-05-07,APPROVED_FOR_COMPOSITE_SCORING
4,defensive_momentum_reversal_blend,2,2098,478,0.894402,0.105598,2018-02-22,2026-05-07,APPROVED_FOR_COMPOSITE_SCORING
5,balanced_stress_alpha,3,2098,478,0.894766,0.105234,2018-02-21,2026-05-07,APPROVED_FOR_COMPOSITE_SCORING
3,defensive_reversal_blend,3,2098,478,0.894766,0.105234,2018-02-21,2026-05-07,APPROVED_FOR_COMPOSITE_SCORING
1,stress_pullback_blend_20,2,2098,478,0.894766,0.105234,2018-02-21,2026-05-07,APPROVED_FOR_COMPOSITE_SCORING
0,stress_reversal_blend_20,2,2098,478,0.894766,0.105234,2018-02-21,2026-05-07,APPROVED_FOR_COMPOSITE_SCORING
7,short_horizon_reversal_blend,3,2098,478,0.901313,0.098687,2018-01-31,2026-05-07,APPROVED_FOR_COMPOSITE_SCORING
2,vol_reversal_short_blend,3,2098,478,0.901313,0.098687,2018-01-31,2026-05-07,APPROVED_FOR_COMPOSITE_SCORING


status
APPROVED_FOR_COMPOSITE_SCORING    8
Name: composite_count, dtype: int64

## 8. Save outputs to SQLite

In [7]:
saved_paths = save_composite_outputs(
    composite_signals=composite_signals,
    metadata=composite_metadata,
    quality=composite_quality,
    db_path=sqlite_db_path,
    run_id=run_id,
    composite_version=COMPOSITE_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            'artifact': artifact,
            'current_table': tables[0],
            'history_table': tables[1],
            'sqlite_path': str(saved_paths[artifact]),
        }
        for artifact, tables in COMPOSITE_TABLES.items()
    ]
)

display(sqlite_tables_written)

,artifact,current_table,history_table,sqlite_path
0,signals,composite_signals_current,composite_signals_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,composite_metadata_current,composite_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,composite_quality_current,composite_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 9. Final summary

In [8]:
all_nan_composites = [name for name, panel in composite_signals.items() if panel.isna().all().all()]

summary = pd.DataFrame(
    [
        {'metric': 'run_id', 'value': run_id},
        {'metric': 'run_timestamp', 'value': run_timestamp},
        {'metric': 'composite_version', 'value': COMPOSITE_VERSION},
        {'metric': 'n_composites', 'value': len(composite_signals)},
        {'metric': 'n_unique_components', 'value': len(component_names)},
        {'metric': 'all_nan_composites', 'value': ', '.join(all_nan_composites)},
    ]
)

print('Composite summary')
display(summary)

print('Component list')
display(pd.Series(component_names, name='component_signal').to_frame())

print('Quality status counts')
display(quality_status_counts.rename('composite_count'))

print('SQLite tables written')
display(sqlite_tables_written)

Composite summary


,metric,value
0,run_id,phase2_nb02b_composite_20260507_225304
1,run_timestamp,2026-05-07 22:53:04
2,composite_version,phase2_composite_v1
3,n_composites,8
4,n_unique_components,9
5,all_nan_composites,


Component list


,component_signal
0,distance_from_ma_20
1,low_vol_strength
2,mean_reversion_20
3,momentum_20
4,momentum_60
5,reversal_10d
6,reversal_5d
7,volatility_20
8,volatility_60


Quality status counts


status
APPROVED_FOR_COMPOSITE_SCORING    8
Name: composite_count, dtype: int64

SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,signals,composite_signals_current,composite_signals_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,composite_metadata_current,composite_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,composite_quality_current,composite_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
